<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/SourceCodeOfUniverse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 7.3 MB/s eta 0:00:00


In [4]:

# ==============================================================================
#  The AI Calibration Forge: A direct search for the source code of the universe.
#
#  Objective: Evolve the complete, disordered laws of physics to simultaneously:
#  1. Match the observed lepton mass ratios (Accuracy).
#  2. Minimize the complexity of the underlying algorithm (Elegance).
# ==============================================================================

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp
from scipy.linalg import expm
import cma
import time

# --- Physical Constants (Target for Accuracy) ---
TARGET_RATIO_MUON_ELECTRON = 105.658 / 0.511
TARGET_RATIO_TAU_ELECTRON = 1776.86 / 0.511

# --- Universe Parameters ---
FORGE_UNIVERSE_SIZE = 6
GENES_PER_INTERACTION = 15
NUM_UNIVERSE_GENES = FORGE_UNIVERSE_SIZE * GENES_PER_INTERACTION

# --- The "Elegance" Parameter ---
# This is the weighting factor 'w'. It controls how much the AI
# prioritizes simplicity vs. accuracy. A higher value forces a simpler solution.
# We start with a small value to let the AI explore freely first.
ELEGANCE_WEIGHT = 0.01

# ==============================================================================
#  STEP 1: Build a Universe (Unchanged)
# ==============================================================================

PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

def build_aoe_from_full_genome(genome: np.ndarray, n_qubits: int) -> np.ndarray:
    if n_qubits < 2: return np.identity(2**n_qubits)
    full_genome = genome.reshape((n_qubits, GENES_PER_INTERACTION))
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        genes_for_this_interaction = full_genome[i]
        generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=genes_for_this_interaction)
        u_gate_matrix = expm(-1j * generator_h.to_matrix())
        u_gate_op = Operator(u_gate_matrix)
        qc.append(u_gate_op, [i, (i + 1) % n_qubits])
    return Operator(qc).data

# ==============================================================================
#  STEP 2: The NEW, Calibrated Fitness Function
# ==============================================================================

def get_energy_spectrum(unitary: np.ndarray) -> list:
    eigenvalues = np.linalg.eigvalsh(unitary)
    energies = []
    for eigval in eigenvalues:
        phase = np.angle(eigval)
        energy = np.abs(phase)
        if not (np.isclose(energy, 0.0) or np.isclose(energy, 2 * np.pi)):
            energies.append(energy)
    unique_energies = []
    energies.sort()
    if energies:
        unique_energies.append(energies[0])
        for i in range(1, len(energies)):
            if not np.isclose(energies[i], energies[i-1], rtol=1e-4):
                unique_energies.append(energies[i])
    return unique_energies

def calibrated_fitness_function(genome: np.ndarray) -> float:
    """
    The new fitness function with two components: accuracy and elegance.
    The AI's goal is to minimize this combined cost.
    """
    try:
        # --- Accuracy Cost ---
        U_aoe = build_aoe_from_full_genome(genome, FORGE_UNIVERSE_SIZE)
        energies = get_energy_spectrum(U_aoe)

        if len(energies) < 3: return 1000.0

        e_electron, e_muon, e_tau = energies[0], energies[1], energies[2]

        if np.isclose(e_electron, 0.0): return 1000.0

        predicted_ratio_muon = e_muon / e_electron
        predicted_ratio_tau = e_tau / e_electron

        error_muon = ((predicted_ratio_muon - TARGET_RATIO_MUON_ELECTRON) / TARGET_RATIO_MUON_ELECTRON)**2
        error_tau = ((predicted_ratio_tau - TARGET_RATIO_TAU_ELECTRON) / TARGET_RATIO_TAU_ELECTRON)**2
        cost_ratio = np.log1p(error_muon + error_tau)

        # --- Elegance Cost ---
        # We define elegance as sparsity. A simpler algorithm uses fewer "parts."
        # The L1 norm (sum of absolute values of the genes) is a perfect measure of this.
        # A universe with smaller, sparser laws is more "elegant."
        cost_efficiency = np.sum(np.abs(genome))

        # --- Total Cost ---
        total_cost = cost_ratio + (ELEGANCE_WEIGHT * cost_efficiency)
        return total_cost

    except Exception:
        return 2000.0

# ==============================================================================
#  STEP 3: Run the AI Calibration Forge
# ==============================================================================

if __name__ == "__main__":
    print(f"--- The AI Calibration Forge ---")
    print(f"Objective: Evolve a {FORGE_UNIVERSE_SIZE}-qubit disordered AoE to match lepton ratios")
    print("           while simultaneously maximizing for algorithmic elegance.")
    print(f"Searching a genetic space of {NUM_UNIVERSE_GENES} parameters.")

    print("\n--- Starting AI Forge to Reverse-Engineer the Laws of Physics ---")

    x0 = np.random.uniform(-np.pi, np.pi, NUM_UNIVERSE_GENES)
    sigma0 = 0.5
    options = {'bounds': [-np.pi, np.pi], 'maxfevals': 20000, 'verbose': -9}

    es = cma.CMAEvolutionStrategy(x0, sigma0, options)

    start_time = time.time()
    last_print_time = start_time

    while not es.stop():
        solutions = es.ask()
        fitnesses = [calibrated_fitness_function(s) for s in solutions]
        es.tell(solutions, fitnesses)

        current_time = time.time()
        if current_time - last_print_time > 5:
             print(f"  > Iteration #{es.countiter}, Best Total Cost: {es.result.fbest:.6f}", end='\r')
             last_print_time = current_time

    end_time = time.time()
    print(f"\n  > Forge complete in {end_time - start_time:.2f}s.")

    champion_genome = es.result.xbest

    # ==============================================================================
    #  STEP 4: Analyze the Champion Universe
    # ==============================================================================

    print("\n--- Final Analysis of the Champion Universe ---")

    final_U_aoe = build_aoe_from_full_genome(champion_genome, FORGE_UNIVERSE_SIZE)
    final_energies = get_energy_spectrum(final_U_aoe)

    def analyze_and_report(energies: list, genome: np.ndarray):
        print("\n--- ACCURACY REPORT ---")
        if not energies or len(energies) < 3:
            print("The champion universe did not produce enough energy levels for a full comparison.")
            return

        e_electron, e_muon, e_tau = energies[0], energies[1], energies[2]
        predicted_ratio_muon = e_muon / e_electron
        predicted_ratio_tau = e_tau / e_electron

        print(f"E1 (Electron): {e_electron:.4f} | E2 (Muon): {e_muon:.4f} | E3 (Tau): {e_tau:.4f}")
        print("-" * 50)
        print(f"Predicted Muon/Electron Ratio: {predicted_ratio_muon:.2f} (Actual: {TARGET_RATIO_MUON_ELECTRON:.2f})")
        print(f"Predicted Tau/Electron Ratio:  {predicted_ratio_tau:.2f} (Actual: {TARGET_RATIO_TAU_ELECTRON:.2f})")

        print("\n--- ELEGANCE REPORT ---")
        elegance_score = np.sum(np.abs(genome))
        print(f"Algorithmic Complexity (L1 Norm): {elegance_score:.2f}")

        print("\n--- SOURCE CODE OF THE UNIVERSE (Top 5 Genes) ---")
        print("The most significant components of the fundamental interaction laws:")
        sorted_genes = sorted(zip(PAULI_BASIS_2Q, genome.flatten()), key=lambda item: abs(item[1]), reverse=True)
        # Note: This analysis is simplified as it averages over all 6 interactions.
        # A deeper analysis would look at the genes for each of the 6 points in space.

        # We need to reshape the genome to analyze it properly
        full_genome_reshaped = genome.reshape((FORGE_UNIVERSE_SIZE, GENES_PER_INTERACTION))
        avg_genome = np.mean(np.abs(full_genome_reshaped), axis=0)

        avg_sorted_genes = sorted(zip(PAULI_BASIS_2Q, avg_genome), key=lambda item: item[1], reverse=True)
        for pauli, coeff_magnitude in avg_sorted_genes[:5]:
            print(f"  Avg. |{pauli}| contribution: {coeff_magnitude:.4f}")

    analyze_and_report(final_energies, champion_genome)

--- The AI Calibration Forge ---
Objective: Evolve a 6-qubit disordered AoE to match lepton ratios
           while simultaneously maximizing for algorithmic elegance.
Searching a genetic space of 90 parameters.

--- Starting AI Forge to Reverse-Engineer the Laws of Physics ---

  > Forge complete in 0.95s.

--- Final Analysis of the Champion Universe ---

--- ACCURACY REPORT ---
The champion universe did not produce enough energy levels for a full comparison.
